In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
sample_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/sample_submission.csv")

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
sample_df

In [ ]:
def analyze_dataframe(df):
    """
    データフレームを分析し、欠損値と数値変換エラーの可能性を報告する関数。
    """
    print("✅ データフレームの読み込みに成功しました。\n")
    
    # --- 1. 欠損値のチェック ---
    print("--- 欠損値のチェック ---")
    missing_values = df.isnull().sum()
    missing_values = missing_values[missing_values > 0]

    if not missing_values.empty:
        print("以下の列に欠損値が見つかりました。")
        # 欠損値の数が多い順に並び替えて表示
        print(missing_values.sort_values(ascending=False))
    else:
        print("欠損値は見つかりませんでした。")

    print("\n" + "="*50 + "\n")


    # --- 2. 数値変換エラーのチェック ---
    print("--- 数値変換エラーのチェック ---")
    potential_errors = {}
    # オブジェクト型（主に文字列）の列を対象にループ
    for col in df.select_dtypes(include=['object']).columns:
        # 数値に変換しようと試み、変換できないものをNaNにする
        numeric_col = pd.to_numeric(df[col], errors='coerce')
        
        # 元のデータには値があるが、数値に変換するとNaNになるものの数を数える
        # (元のNaNと区別するため)
        error_count = numeric_col.isnull().sum() - df[col].isnull().sum()
        
        if error_count > 0:
            # エラーの原因となっている可能性のあるユニークな値を取得
            # isna() と isnull() は同じ
            error_values = df[col][numeric_col.isnull() & df[col].notnull()].unique()
            potential_errors[col] = {
                'count': error_count,
                'example_values': error_values[:5] # 最初の5つのユニークなエラー値を表示
            }

    if potential_errors:
        print("以下の【オブジェクト型】の列に、数値に変換できない値が含まれている可能性があります。")
        print("（これらが意図したカテゴリ変数の場合は、エラー修正の必要はありません）")
        for col, details in potential_errors.items():
            print(f"\n- 列名: '{col}'")
            print(f"  数値に変換できない値の数: {details['count']}")
            print(f"  値の例: {details['example_values']}")
    else:
        print("オブジェクト型の列で、数値に変換できないような値は見つかりませんでした。")

    print("\n" + "="*50 + "\n")
    print("分析が完了しました。")
    print("この結果を基に、データクレンジング（エラー修正）の計画を立てることができます。")

In [ ]:
#下記は元からあったもの
X = train_df[["LotArea","YrSold","OverallQual","WoodDeckSF"]]
y = train_df["SalePrice"]
X_test = test_df[["LotArea","YrSold","OverallQual","WoodDeckSF"]]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

### モデル構築

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression()

In [ ]:
model.fit(X_train, y_train)

### モデル適用と評価

In [ ]:
pred_train = model.predict(X_train)
pred_valid = model.predict(X_valid)
pred_test = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
print(mean_squared_error(y_train, pred_train), mean_squared_error(y_valid, pred_valid))

### Submission

In [ ]:
sub_df = pd.DataFrame({
    "Id": sample_df["Id"], 
    "SalePrice": pred_test
})

In [ ]:
sub_df.to_csv("simple.csv", index=False)

In [ ]:
sub_df

In [ ]:
#ここからはオリジナルコード

In [ ]:
def auto_clean_for_kaggle(train_df, test_df):

    print("--- とりあえず全部のデータを自動で修正処理を行う ---")
    
    df_list = [train_df, test_df]
    
    # --- 1. 欠損値の自動処理 ---
    print("\n[ステップ1] 欠損値の処理を開始...")
    # 学習・テスト両方から欠損列を特定
    missing_in_train = [col for col in train_df.columns.drop('SalePrice', errors='ignore') if train_df[col].isnull().sum() > 0]
    missing_in_test = [col for col in test_df.columns if test_df[col].isnull().sum() > 0]
    all_missing_cols = list(set(missing_in_train + missing_in_test))

    if not all_missing_cols:
        print("-> 欠損値のある列は見つかりませんでした。")
    else:
        print(f"-> 対象列 (計{len(all_missing_cols)}列) の処理を開始...")
        for col in all_missing_cols:
            for df in df_list:
                if col in df.columns:
                    # 欠損フラグ列を追加
                    flag_col_name = f"{col}_is_missing"
                    df[flag_col_name] = df[col].isnull()
                    # 0で埋める
                    df[col].fillna(0, inplace=True)
    print("-> 欠損フラグの追加と0での補完が完了しました。")

    # --- 2. 外れ値の自動処理 (Clipping) ---
    print("\n[ステップ2] 外れ値の処理を開始...")
    numerical_cols = train_df.select_dtypes(include=np.number).columns.tolist()
    cols_to_exclude = ['Id', 'MSSubClass', 'SalePrice'] + [col for col in train_df.columns if '_is_missing' in col] 
    target_numerical_cols = [col for col in numerical_cols if col not in cols_to_exclude]
    
    print(f"-> {len(target_numerical_cols)}個の数値列を対象に処理します。")
    for col in target_numerical_cols:
        # 閾値は学習データから計算
        q01 = train_df[col].quantile(0.01)
        q99 = train_df[col].quantile(0.99)
        for df in df_list:
            if col in df.columns:
                df[col] = df[col].clip(lower=q01, upper=q99)
    print("-> 外れ値の丸め処理 (Clipping) が完了しました。")
    
    print("\n--- 全ての自動修正処理が完了しました ---")
    return train_df, test_df

In [ ]:
try:
    # --- データの読み込み ---
    train_df_original = train_df
    test_df_original = test_df
    sample_df = sample_df
    
    print("✅ 'train.csv', 'test.csv', 'sample_submission.csv' の読み込みに成功しました。\n")

except FileNotFoundError as e:
    print(f"❌ エラー: ファイルが見つかりません。{e}")
    print("必要なCSVファイルがColab環境にアップロードされているか確認してください。")
except Exception as e:
    print(f"予期せぬエラーが発生しました: {e}")

In [ ]:
# --- クレンジング関数の実行 ---
train_processed, test_processed = auto_clean_for_kaggle(
    train_df_original.copy(), 
    test_df_original.copy()
)
print("\n" + "="*50)
print(" === クレンジング処理が完了しました === ")
print("="*50 + "\n")


# --- X, y, X_test の作成 ---

# ★ 目的変数 y を対数変換する ★
y = np.log1p(train_processed["SalePrice"])

# 特徴量 X からは Id と SalePrice を除外
feature_columns = train_processed.drop(columns=['SalePrice', 'Id']).columns.tolist()
X = train_processed[feature_columns]

# テストデータも X と同じ列構成にする
final_test_columns = [col for col in feature_columns if col in test_processed.columns]
X_test = test_processed[final_test_columns]

print("✅ X, y (対数変換済み), X_test の準備が完了しました。\n")

In [ ]:
# --- One-Hot Encoding (カテゴリ変数の数値化) ---
print("--- One-Hot Encoding を開始します ---")
X_encoded = pd.get_dummies(X, dummy_na=False) 
X_test_encoded = pd.get_dummies(X_test, dummy_na=False)

# 学習データとテストデータで列を揃える (Kaggle の必須テクニック)
X_final, X_test_final = X_encoded.align(X_test_encoded, join='inner', axis=1, fill_value=0)

print(f"✅ One-Hot Encoding が完了しました。（特徴量: {X_final.shape[1]}個）\n")

# --- モデルの学習と評価 ---
print("--- モデルの学習と評価を開始します ---")

# データを学習用と検証用に分割
X_train, X_valid, y_train, y_valid = train_test_split(X_final, y, random_state=0)

# モデルを定義し、学習
model = LinearRegression()
model.fit(X_train, y_train)
print("✅ モデルの学習 (fit) が完了しました。")

# 検証データで評価 (対数誤差 RMSLE)
pred_valid = model.predict(X_valid)
rmse_valid_log = np.sqrt(mean_squared_error(y_valid, pred_valid))
print(f"\n検証データのRMSLE (対数誤差): {rmse_valid_log:.4f}")

In [ ]:
# --- 提出用ファイルの作成 ---
print("\n--- 提出用ファイルの作成を開始します ---")

# テストデータで予測 (この時点ではまだ対数スケール)
pred_test_log = model.predict(X_test_final) 

# ★ np.expm1 を使って、対数スケールの予測値を元の価格スケールに戻す ★
submission_predictions = np.expm1(pred_test_log)

# 提出用DataFrameを作成
sub_df = pd.DataFrame({
    "Id": sample_df["Id"], # sample_submission.csv のID
    "SalePrice": submission_predictions # 元の価格に戻した予測値
})

# CSVファイルとして保存
sub_df.to_csv("submission.csv", index=False)

print("✅ 'submission.csv' の作成が完了しました。")
print("作成されたファイル（先頭5行）:")
print(sub_df.head())

In [ ]:
#10月24日の講義を受けて　文字列を削除するステップを入れたら・・・点数が下がった。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso # ★ Lassoを追加
from sklearn.metrics import mean_squared_error

def auto_clean_for_kaggle(train_df, test_df):
    """
    Kaggle向けに、学習データとテストデータを一貫して処理する関数。
    """
    print("--- とりあえず全部のデータを自動で修正処理を行う ---")
    
    df_list = [train_df, test_df]
    
    # --- 1. 欠損値の自動処理 ---
    print("\n[ステップ1] 欠損値の処理を開始...")
    missing_in_train = [col for col in train_df.columns.drop('SalePrice', errors='ignore') if train_df[col].isnull().sum() > 0]
    missing_in_test = [col for col in test_df.columns if test_df[col].isnull().sum() > 0]
    all_missing_cols = list(set(missing_in_train + missing_in_test))

    if not all_missing_cols:
        print("-> 欠損値のある列は見つかりませんでした。")
    else:
        print(f"-> 対象列 (計{len(all_missing_cols)}列) の処理を開始...")
        for col in all_missing_cols:
            for df in df_list:
                if col in df.columns:
                    flag_col_name = f"{col}_is_missing"
                    df[flag_col_name] = df[col].isnull()
                    df[col].fillna(0, inplace=True)
    print("-> 欠損フラグの追加と0での補完が完了しました。")

    # --- 2. 外れ値の自動処理 (Clipping) ---
    print("\n[ステップ2] 外れ値の処理を開始...")
    numerical_cols = train_df.select_dtypes(include=np.number).columns.tolist()
    cols_to_exclude = ['Id', 'MSSubClass', 'SalePrice'] + [col for col in train_df.columns if '_is_missing' in col] 
    target_numerical_cols = [col for col in numerical_cols if col not in cols_to_exclude]
    
    print(f"-> {len(target_numerical_cols)}個の数値列を対象に処理します。")
    for col in target_numerical_cols:
        q01 = train_df[col].quantile(0.01)
        q99 = train_df[col].quantile(0.99)
        for df in df_list:
            if col in df.columns:
                df[col] = df[col].clip(lower=q01, upper=q99)
    print("-> 外れ値の丸め処理 (Clipping) が完了しました。")
    
    print("\n--- 全ての自動修正処理が完了しました ---")
    return train_df, test_df

In [ ]:
try:
    # --- データの読み込み ---
    # (Kaggle環境では、'/kaggle/input/...' などのパスから読み込みます)
    train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
    test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
    sample_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/sample_submission.csv")
    
    print("✅ 'train.csv', 'test.csv', 'sample_submission.csv' の読み込みに成功しました。\n")

except FileNotFoundError as e:
    print(f"❌ エラー: ファイルが見つかりません。{e}")
    print("必要なCSVファイルがColab環境にアップロードされているか確認してください。")
except Exception as e:
    print(f"予期せぬエラーが発生しました: {e}")

In [ ]:
# --- クレンジング関数の実行 ---
train_processed, test_processed = auto_clean_for_kaggle(
    train_df.copy(), 
    test_df.copy()
)
print("\n" + "="*50)
print(" === クレンジング処理が完了しました === ")
print("="*50 + "\n")


# --- X, y, X_test の作成 ---

# ★ 目的変数 y を対数変換する ★
y = np.log1p(train_processed["SalePrice"])

# ==========================================================
# ★★★ 特徴量選択を削除 ★★★
# 'Neighborhood' なども含む、IdとSalePrice以外の「全て」を特徴量として使います
# ==========================================================
feature_columns = train_processed.drop(columns=['SalePrice', 'Id']).columns.tolist()
X = train_processed[feature_columns]

# テストデータも X と同じ列構成にする
final_test_columns = [col for col in feature_columns if col in test_processed.columns]
X_test = test_processed[final_test_columns]

print(f"✅ X (特徴量 {len(feature_columns)}個), y (対数変換済み), X_test の準備が完了しました。\n")

In [ ]:
# --- One-Hot Encoding (カテゴリ変数の数値化) ---
print("--- One-Hot Encoding を開始します ---")
X_encoded = pd.get_dummies(X, dummy_na=False) 
X_test_encoded = pd.get_dummies(X_test, dummy_na=False)

# 学習データとテストデータで列を揃える
X_final, X_test_final = X_encoded.align(X_test_encoded, join='inner', axis=1, fill_value=0)

print(f"✅ One-Hot Encoding が完了しました。（特徴量: {X_final.shape[1]}個）\n")

# --- モデルの学習と評価 ---
print("--- モデルの学習と評価を開始します ---")

# データを学習用と検証用に分割
X_train, X_valid, y_train, y_valid = train_test_split(X_final, y, random_state=0)

# ==========================================================
# ★★★ モデルを Lasso に変更 ★★★
# alphaは「Lassoの強さ（正則化の強さ）」です。0.0005は良い出発点です。
# max_iterは計算回数の上限を増やし、計算が収束しやすくするためのおまじないです。
# ==========================================================
model = Lasso(alpha=0.0005, random_state=0, max_iter=2000)
# ==========================================================

model.fit(X_train, y_train)
print("✅ モデルの学習 (fit) が完了しました。")

# 検証データで評価 (対数誤差 RMSLE)
pred_valid = model.predict(X_valid)
rmse_valid_log = np.sqrt(mean_squared_error(y_valid, pred_valid))
print(f"\n【検証スコア】 検証データのRMSLE (対数誤差): {rmse_valid_log:.5f}")
print(f"（目標スコア: 0.16437 より良くなる（小さくなる）ことを目指します）")

In [ ]:
X_train

In [ ]:
# --- 提出用ファイルの作成 ---
print("\n--- 提出用ファイルの作成を開始します ---")

# ==========================================================
# ★★★ 全データで再学習（モデルを Lasso に変更） ★★★
# ==========================================================
print("Kaggle提出用に、全ての学習データ(100%)を使ってモデルを再学習します...")

# ★ モデルを Lasso に変更 (検証時と全く同じ設定) ★
final_model = Lasso(alpha=0.0005, random_state=0, max_iter=2000) 

# X_final (全学習データ) と y (全正解データ) で学習
final_model.fit(X_final, y) 
print("✅ 全データでの再学習が完了しました。")

# 再学習させた「final_model」で予測
pred_test_log = final_model.predict(X_test_final) 
# ==========================================================

# ★ np.expm1 を使って、対数スケールの予測値を元の価格スケールに戻す ★
submission_predictions = np.expm1(pred_test_log)

# (Lassoの予測結果が稀にマイナスになることがあるため、念のため0未満は0にします)
submission_predictions[submission_predictions < 0] = 0

# 提出用DataFrameを作成
sub_df = pd.DataFrame({
    "Id": sample_df["Id"], # sample_submission.csv のID
    "SalePrice": submission_predictions # 元の価格に戻した予測値
})

# CSVファイルとして保存
sub_df.to_csv("submission2.csv", index=False)

print("✅ 'submission2.csv' の作成が完了しました。")
print("作成されたファイル（先頭5行）:")
print(sub_df.head())